# Metadata & Filtering (Lightweight)

**Purpose:** Teach that real retrieval is not only vectors — **metadata matters**.

Semantic search systems in production almost always use **both**:
- vectors for meaning-based similarity
- metadata for scoping, policy, freshness, and precision

In this notebook we will:
1. Explain why metadata exists (date, category, source, permissions, etc.)
2. Attach metadata to each document
3. Do **two-stage retrieval**:
   - filter by metadata (pre-filter or post-filter)
   - then vector search
4. Discuss tradeoffs:
   - filtering often improves **precision**
   - but may reduce **recall**
5. Show outputs:
   - results **with** and **without** filters
   - “scoped queries” that become more relevant with filtering


## 0) Setup

We’ll use:
- `sentence-transformers` for embeddings
- `faiss` for nearest neighbor search
- `pandas` for nice tables


In [1]:
# Install dependencies if needed (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

try:
    import faiss  # noqa: F401
except ImportError:
    !uv add faiss-cpu

In [2]:
import random
import numpy as np
import pandas as pd
import faiss

from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 150)
random.seed(42)
np.random.seed(42)

## 1) Why metadata exists

Embeddings capture **meaning**. Metadata captures **context** and **constraints**.

Common metadata fields:
- **date** (freshness, recency filtering)
- **category/topic** (scope by domain)
- **source** (trustworthiness, provenance)
- **permissions** (who is allowed to see it)
- **document type** (policy, FAQ, ticket, log)
- **owner/team** (routing)

In many real systems:
> You *must* apply metadata filters before retrieval to prevent returning restricted results.


## 2) Build a small document set + metadata

We’ll create a dataset with:
- `text`
- `category` (finance/software/weather/health/business)
- `source` (handbook/wiki/support/newsletter)
- `date` (simulated recency)

This will let us demonstrate how filtering improves relevance for scoped queries.


In [3]:
categories = {
    "finance": [
        "Reduce spending by reviewing subscriptions and recurring bills.",
        "Banks assess credit risk before approving loans.",
        "Diversify investments to manage portfolio risk.",
        "Build an emergency fund by setting aside a small amount weekly.",
        "Pay down high-interest debt first to save on interest payments.",
    ],
    "software": [
        "Refactor code to reduce technical debt and improve maintainability.",
        "Profile your program to find performance bottlenecks.",
        "Add caching to avoid recomputing expensive results.",
        "Use indexes in databases to speed up query execution.",
        "Improve reliability by adding retries and timeouts.",
    ],
    "weather": [
        "Monitor official advisories when a typhoon is nearby.",
        "Avoid driving through flooded roads during heavy rain.",
        "Storm surge can be more dangerous than wind in coastal zones.",
        "Prepare emergency supplies like water, food, and batteries.",
        "Secure loose objects before severe weather arrives.",
    ],
    "health": [
        "Breathing exercises can reduce stress in the short term.",
        "Walking daily can improve cardiovascular health.",
        "Prioritize sleep to support focus and memory.",
        "Balance meals with protein, fiber, and healthy fats.",
        "Avoid sugary drinks to reduce empty calories.",
    ],
    "business": [
        "Customer retention improves when support resolves issues quickly.",
        "A loyalty program can increase repeat purchases.",
        "Segment users to tailor messaging to their needs.",
        "Measure campaign impact with controlled experiments.",
        "Forecast demand to prevent stockouts during peak seasons.",
    ],
}

sources = ["handbook", "wiki", "support", "newsletter"]

def make_docs(repeats=8):
    rows = []
    base_date = datetime(2026, 1, 29)  # align to your course date context
    for cat, texts in categories.items():
        for _ in range(repeats):
            for t in texts:
                src = random.choice(sources)
                # simulate docs over last 120 days
                days_ago = random.randint(0, 120)
                dt = base_date - timedelta(days=days_ago)
                rows.append({
                    "category": cat,
                    "source": src,
                    "date": dt.date().isoformat(),
                    "text": t
                })
    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    df.insert(0, "doc_id", [f"D{i:04d}" for i in range(len(df))])
    return df

df = make_docs(repeats=10)
df.shape, df.head()

((246, 5),
   doc_id category    source        date  \
 0  D0000  finance  handbook  2026-01-26   
 1  D0001  finance   support  2025-12-29   
 2  D0002  finance      wiki  2026-01-12   
 3  D0003  finance  handbook  2025-11-04   
 4  D0004  finance  handbook  2025-11-15   
 
                                                               text  
 0  Reduce spending by reviewing subscriptions and recurring bills.  
 1                 Banks assess credit risk before approving loans.  
 2                  Diversify investments to manage portfolio risk.  
 3  Build an emergency fund by setting aside a small amount weekly.  
 4  Pay down high-interest debt first to save on interest payments.  )

## 3) Build embeddings + FAISS index (global)

We build **one global index** for all documents.

Important design idea:
- FAISS stores vectors; metadata stays in your dataframe / database.
- Search returns indices → you map indices back → then apply metadata logic.

We will use normalized embeddings + inner product (cosine-like).


In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

texts = df["text"].tolist()
emb = model.encode(texts, normalize_embeddings=True).astype("float32")

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

print("Docs:", len(df), "| Embeddings:", emb.shape, "| Index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Docs: 246 | Embeddings: (246, 384) | Index size: 246


## 4) Retrieval functions

We’ll implement:
- **Unfiltered semantic search**: pure vector similarity
- **Post-filter**: retrieve top-N (e.g., 50), then filter by metadata, then take top-k
- **Pre-filter**: build a temporary index over only the filtered subset (simple but slower to build per query)

In production, you typically do:
- pre-filter using a metadata store (SQL/Elastic/etc.), then vector search in a narrowed set
- or post-filter if constraints are light and top-N is large enough

We’ll demo both, but default to **post-filter** because it’s easier to teach.


In [5]:
def semantic_search(query, k=5):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    rows = []
    for rank, (i, score) in enumerate(zip(idx[0], scores[0]), start=1):
        rows.append({
            "rank": rank,
            "doc_id": df.loc[i, "doc_id"],
            "score": float(score),
            "category": df.loc[i, "category"],
            "source": df.loc[i, "source"],
            "date": df.loc[i, "date"],
            "text": df.loc[i, "text"],
        })
    return pd.DataFrame(rows)

def semantic_search_postfilter(query, k=5, top_n=50, category=None, source=None, min_date=None):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, top_n)
    cand = df.iloc[idx[0]].copy()
    cand = cand.assign(score=scores[0])
    
    # Apply filters
    if category is not None:
        cand = cand[cand["category"] == category]
    if source is not None:
        cand = cand[cand["source"] == source]
    if min_date is not None:
        cand = cand[cand["date"] >= min_date]
    
    cand = cand.sort_values("score", ascending=False).head(k).reset_index(drop=True)
    cand.insert(0, "rank", np.arange(1, len(cand) + 1))
    return cand[["rank", "doc_id", "score", "category", "source", "date", "text"]]

def semantic_search_prefilter(query, k=5, category=None, source=None, min_date=None):
    # Filter dataframe first
    subset = df.copy()
    if category is not None:
        subset = subset[subset["category"] == category]
    if source is not None:
        subset = subset[subset["source"] == source]
    if min_date is not None:
        subset = subset[subset["date"] >= min_date]
        
    if len(subset) == 0:
        return pd.DataFrame(columns=["rank", "doc_id", "score", "category", "source", "date", "text"])
    
    # Build a temporary index over subset embeddings
    sub_idx = subset.index.to_numpy()
    sub_emb = emb[sub_idx]
    dim = sub_emb.shape[1]
    tmp_index = faiss.IndexFlatIP(dim)
    tmp_index.add(sub_emb)
    
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, local_idx = tmp_index.search(q, min(k, len(subset)))
    
    rows = []
    for rank, (li, score) in enumerate(zip(local_idx[0], scores[0]), start=1):
        global_i = sub_idx[int(li)]
        rows.append({
            "rank": rank,
            "doc_id": df.loc[global_i, "doc_id"],
            "score": float(score),
            "category": df.loc[global_i, "category"],
            "source": df.loc[global_i, "source"],
            "date": df.loc[global_i, "date"],
            "text": df.loc[global_i, "text"],
        })
    return pd.DataFrame(rows)

# Sanity check
semantic_search("How can I reduce spending?", k=5)

,rank,doc_id,score,category,source,date,text
0,1,D0020,0.707896,finance,handbook,2025-12-15,Reduce spending by reviewing subscriptions and recurring bills.
1,2,D0015,0.707896,finance,newsletter,2025-12-17,Reduce spending by reviewing subscriptions and recurring bills.
2,3,D0010,0.707896,finance,newsletter,2026-01-01,Reduce spending by reviewing subscriptions and recurring bills.
3,4,D0005,0.707896,finance,newsletter,2026-01-25,Reduce spending by reviewing subscriptions and recurring bills.
4,5,D0000,0.707896,finance,handbook,2026-01-26,Reduce spending by reviewing subscriptions and recurring bills.


## 5) Demo: results with and without filters

We’ll use **scoped queries** where the user implies a filter:

- “*In software engineering…*” → category=software
- “*Latest guidance…*” → min_date filter
- “*From the handbook…*” → source filter

We’ll show:
1. Unfiltered vector search
2. Post-filtered search (recommended teaching pattern)
3. Pre-filtered search (conceptual alternative)


In [6]:
def show_comparison(query, k=5, category=None, source=None, min_date=None, top_n=50):
    print("QUERY:", query)
    print("FILTERS:", {"category": category, "source": source, "min_date": min_date})
    
    print("\n--- Unfiltered (pure semantic) ---")
    display(semantic_search(query, k=k))
    
    print("\n--- Post-filter (retrieve top-N then filter) ---")
    display(semantic_search_postfilter(query, k=k, top_n=top_n, category=category, source=source, min_date=min_date))
    
    print("\n--- Pre-filter (filter then build/search subset index) ---")
    display(semantic_search_prefilter(query, k=k, category=category, source=source, min_date=min_date))

# Example 1: scoped to software
show_comparison(
    query="How do I make my service more reliable when network calls fail?",
    k=5,
    category="software"
)

QUERY: How do I make my service more reliable when network calls fail?
FILTERS: {'category': 'software', 'source': None, 'min_date': None}

--- Unfiltered (pure semantic) ---


,rank,doc_id,score,category,source,date,text
0,1,D0072,0.474002,software,support,2025-12-05,Improve reliability by adding retries and timeouts.
1,2,D0067,0.474002,software,support,2025-11-20,Improve reliability by adding retries and timeouts.
2,3,D0062,0.474002,software,wiki,2025-10-20,Improve reliability by adding retries and timeouts.
3,4,D0057,0.474002,software,newsletter,2025-12-14,Improve reliability by adding retries and timeouts.
4,5,D0052,0.474002,software,newsletter,2026-01-11,Improve reliability by adding retries and timeouts.



--- Post-filter (retrieve top-N then filter) ---


,rank,doc_id,score,category,source,date,text
0,1,D0097,0.474002,software,support,2025-12-04,Improve reliability by adding retries and timeouts.
1,2,D0092,0.474002,software,newsletter,2025-11-20,Improve reliability by adding retries and timeouts.
2,3,D0087,0.474002,software,handbook,2025-12-30,Improve reliability by adding retries and timeouts.
3,4,D0082,0.474002,software,handbook,2025-11-14,Improve reliability by adding retries and timeouts.
4,5,D0077,0.474002,software,handbook,2025-10-10,Improve reliability by adding retries and timeouts.



--- Pre-filter (filter then build/search subset index) ---


,rank,doc_id,score,category,source,date,text
0,1,D0072,0.474002,software,support,2025-12-05,Improve reliability by adding retries and timeouts.
1,2,D0067,0.474002,software,support,2025-11-20,Improve reliability by adding retries and timeouts.
2,3,D0062,0.474002,software,wiki,2025-10-20,Improve reliability by adding retries and timeouts.
3,4,D0057,0.474002,software,newsletter,2025-12-14,Improve reliability by adding retries and timeouts.
4,5,D0052,0.474002,software,newsletter,2026-01-11,Improve reliability by adding retries and timeouts.


In [7]:
# Example 2: scoped to weather + recency (latest 30 days)
base_date = datetime(2026, 1, 29).date()
min_date_30 = (base_date - timedelta(days=30)).isoformat()

show_comparison(
    query="What should I prepare when a typhoon is coming?",
    k=5,
    category="weather",
    min_date=min_date_30
)

QUERY: What should I prepare when a typhoon is coming?
FILTERS: {'category': 'weather', 'source': None, 'min_date': '2025-12-30'}

--- Unfiltered (pure semantic) ---


,rank,doc_id,score,category,source,date,text
0,1,D0118,0.666053,weather,newsletter,2025-10-11,Monitor official advisories when a typhoon is nearby.
1,2,D0113,0.666053,weather,newsletter,2025-10-18,Monitor official advisories when a typhoon is nearby.
2,3,D0108,0.666053,weather,support,2026-01-20,Monitor official advisories when a typhoon is nearby.
3,4,D0103,0.666053,weather,wiki,2026-01-29,Monitor official advisories when a typhoon is nearby.
4,5,D0098,0.666053,weather,newsletter,2026-01-14,Monitor official advisories when a typhoon is nearby.



--- Post-filter (retrieve top-N then filter) ---


,rank,doc_id,score,category,source,date,text
0,1,D0143,0.666053,weather,wiki,2026-01-22,Monitor official advisories when a typhoon is nearby.
1,2,D0108,0.666053,weather,support,2026-01-20,Monitor official advisories when a typhoon is nearby.
2,3,D0103,0.666053,weather,wiki,2026-01-29,Monitor official advisories when a typhoon is nearby.
3,4,D0098,0.666053,weather,newsletter,2026-01-14,Monitor official advisories when a typhoon is nearby.
4,5,D0141,0.502176,weather,newsletter,2026-01-10,"Prepare emergency supplies like water, food, and batteries."



--- Pre-filter (filter then build/search subset index) ---


,rank,doc_id,score,category,source,date,text
0,1,D0143,0.666053,weather,wiki,2026-01-22,Monitor official advisories when a typhoon is nearby.
1,2,D0108,0.666053,weather,support,2026-01-20,Monitor official advisories when a typhoon is nearby.
2,3,D0103,0.666053,weather,wiki,2026-01-29,Monitor official advisories when a typhoon is nearby.
3,4,D0098,0.666053,weather,newsletter,2026-01-14,Monitor official advisories when a typhoon is nearby.
4,5,D0131,0.502176,weather,handbook,2026-01-18,"Prepare emergency supplies like water, food, and batteries."


In [8]:
# Example 3: user trusts a source (handbook) for policy-style guidance
show_comparison(
    query="How do we reduce technical debt over time?",
    k=5,
    category="software",
    source="handbook"
)

QUERY: How do we reduce technical debt over time?
FILTERS: {'category': 'software', 'source': 'handbook', 'min_date': None}

--- Unfiltered (pure semantic) ---


,rank,doc_id,score,category,source,date,text
0,1,D0068,0.614731,software,handbook,2025-11-03,Refactor code to reduce technical debt and improve maintainability.
1,2,D0063,0.614731,software,newsletter,2025-11-14,Refactor code to reduce technical debt and improve maintainability.
2,3,D0058,0.614731,software,wiki,2026-01-12,Refactor code to reduce technical debt and improve maintainability.
3,4,D0053,0.614731,software,support,2026-01-12,Refactor code to reduce technical debt and improve maintainability.
4,5,D0048,0.614731,software,support,2026-01-21,Refactor code to reduce technical debt and improve maintainability.



--- Post-filter (retrieve top-N then filter) ---


,rank,doc_id,score,category,source,date,text
0,1,D0088,0.614731,software,handbook,2026-01-19,Refactor code to reduce technical debt and improve maintainability.
1,2,D0068,0.614731,software,handbook,2025-11-03,Refactor code to reduce technical debt and improve maintainability.



--- Pre-filter (filter then build/search subset index) ---


,rank,doc_id,score,category,source,date,text
0,1,D0088,0.614731,software,handbook,2026-01-19,Refactor code to reduce technical debt and improve maintainability.
1,2,D0068,0.614731,software,handbook,2025-11-03,Refactor code to reduce technical debt and improve maintainability.
2,3,D0090,0.234145,software,handbook,2025-10-24,Add caching to avoid recomputing expensive results.
3,4,D0060,0.234145,software,handbook,2025-10-11,Add caching to avoid recomputing expensive results.
4,5,D0087,0.227257,software,handbook,2025-12-30,Improve reliability by adding retries and timeouts.


## 6) Tradeoffs: precision vs recall

### Filtering improves precision
Because you remove off-topic results.

### Filtering may reduce recall
Because you might filter out the truly relevant document:
- wrong category label
- missing metadata
- user chose a too-narrow filter
- relevant doc exists but is older than the cutoff

### Post-filter vs pre-filter
- **Post-filter**
  - Easy to implement
  - Risk: if top-N is too small, you might not retrieve enough candidates to survive filtering
- **Pre-filter**
  - Conceptually clean
  - Can be more expensive if you rebuild indices per query (unless you maintain per-filter indices)

In real systems, pre-filtering is often done with:
- database queries (SQL)
- search engines (Elastic)
- access-control layers
then vector search on the narrowed candidate set.


## 7) Mini exercise

1. Try a query that could belong to multiple categories (e.g., “risk”, “performance”).
2. Compare unfiltered vs filtered results.
3. Increase `top_n` in post-filtering from 50 → 200. Did recall improve?
4. Create a filter that is too strict (e.g., category + source + recent date) and observe how results can vanish.

**Key lesson:** Vector search answers “what is similar?”  
Metadata answers “what is allowed / relevant / in scope?”


## Outputs checklist

- ✅ results with and without filters
- ✅ show improved relevance for scoped queries
- ✅ explain tradeoffs (precision vs recall)
